In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet50, ResNet50_Weights
from torchvision import transforms
from torchvision.ops import box_iou
from pycocotools.coco import COCO
from scipy.optimize import linear_sum_assignment
from PIL import Image
import os
from tqdm import tqdm
import time

# ---------------------------
# Model
# ---------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1024):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.pe = pe.unsqueeze(0)  # [1, max_len, d_model]

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :].to(x.device)


class DETR(nn.Module):
    def __init__(self, num_classes=2, hidden_dim=256, nheads=8, num_encoder_layers=6, num_decoder_layers=3,
                 num_queries=50, backbone_name="resnet50"):
        super().__init__()
        resnet = resnet50(weights=ResNet50_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])
        self.input_proj = nn.Conv2d(2048, hidden_dim, kernel_size=1)
        self.position_encoder = PositionalEncoding(hidden_dim)

        self.transformer = nn.Transformer(
            d_model=hidden_dim,
            nhead=nheads,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            batch_first=True
        )

        self.query_embed = nn.Parameter(torch.rand(num_queries, hidden_dim))
        self.class_embed = nn.Linear(hidden_dim, num_classes + 1)
        self.bbox_embed = nn.Linear(hidden_dim, 4)

    def forward(self, x):
        x = self.backbone(x)
        x = self.input_proj(x)
        b, c, h, w = x.shape
        x = x.flatten(2).permute(0, 2, 1)  # [B, HW, C]
        x = self.position_encoder(x)

        query_embed = self.query_embed.unsqueeze(0).repeat(b, 1, 1)
        tgt = torch.zeros_like(query_embed)

        hs = self.transformer(x, tgt)
        logits = self.class_embed(hs)
        boxes = self.bbox_embed(hs).sigmoid()
        return {'pred_logits': logits, 'pred_boxes': boxes}


# ---------------------------
# Hungarian Matcher
# ---------------------------
class HungarianMatcher(nn.Module):
    def __init__(self, class_cost=1, bbox_cost=5):
        super().__init__()
        self.class_cost = class_cost
        self.bbox_cost = bbox_cost

    def forward(self, outputs, targets):
        indices = []
        for b in range(len(targets)):
            out_prob = outputs["pred_logits"][b].softmax(-1)
            out_bbox = outputs["pred_boxes"][b]
            tgt_ids = targets[b]["labels"]
            tgt_bbox = targets[b]["boxes"]

            cost_class = -out_prob[:, tgt_ids]
            cost_bbox = torch.cdist(out_bbox, tgt_bbox, p=1)
            cost = self.class_cost * cost_class + self.bbox_cost * cost_bbox
            i, j = linear_sum_assignment(cost.detach().cpu().numpy())
            indices.append((torch.as_tensor(i, dtype=torch.int64), torch.as_tensor(j, dtype=torch.int64)))
        return indices


# ---------------------------
# Loss Function
# ---------------------------
def detr_loss(outputs, targets, matcher, num_classes):
    indices = matcher(outputs, targets)
    src_logits = outputs['pred_logits']
    src_boxes = outputs['pred_boxes']

    batch_size, num_queries = src_logits.shape[:2]
    target_classes = torch.full((batch_size, num_queries), num_classes, dtype=torch.int64, device=src_logits.device)

    for i, (src_idx, tgt_idx) in enumerate(indices):
        target_classes[i, src_idx] = targets[i]['labels'][tgt_idx]

    loss_cls = F.cross_entropy(src_logits.transpose(1, 2), target_classes)

    pred_boxes = torch.cat([src_boxes[i][src] for i, (src, _) in enumerate(indices)], dim=0)
    target_boxes = torch.cat([t['boxes'][tgt] for t, (_, tgt) in zip(targets, indices)], dim=0)

    loss_bbox = F.l1_loss(pred_boxes, target_boxes, reduction='mean')
    return loss_cls + loss_bbox


# ---------------------------
# Dataset
# ---------------------------
class CocoDetectionDataset(Dataset):
    def __init__(self, img_folder, ann_file):
        self.coco = COCO(ann_file)
        self.ids = list(sorted(self.coco.imgs.keys()))
        self.root = img_folder
        self.transform = transforms.Compose([
            transforms.Resize((512, 512)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225])
        ])

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        path = self.coco.loadImgs(img_id)[0]['file_name']
        img = Image.open(os.path.join(self.root, path)).convert("RGB")
        img_tensor = self.transform(img)

        w, h = img.size
        boxes = []
        labels = []
        for ann in anns:
            x, y, bw, bh = ann['bbox']
            boxes.append([x/w, y/h, bw/w, bh/h])
            labels.append(ann['category_id'] - 1)  # adjust to 0-based index

        return img_tensor, {
            'boxes': torch.tensor(boxes, dtype=torch.float32),
            'labels': torch.tensor(labels, dtype=torch.int64)
        }

    def __len__(self):
        return len(self.ids)


# ---------------------------
# Training Function
# ---------------------------
def train_fn(data_loader, model, criterion, optimizer, device, scheduler, epoch, matcher):
    model.train()
    scaler = torch.cuda.amp.GradScaler()
    total_loss = 0
    start = time.time()

    progress = tqdm(data_loader, desc=f"Epoch {epoch+1}")
    for imgs, targets in progress:
        imgs = torch.stack(imgs).to(device)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        with torch.cuda.amp.autocast():
            outputs = model(imgs)
            loss = criterion(outputs, targets, matcher, num_classes=2)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        if scheduler:
            scheduler.step()

        total_loss += loss.item()
        progress.set_postfix(loss=loss.item())

    elapsed = time.time() - start
    print(f"[Epoch {epoch+1}] Loss: {total_loss/len(data_loader):.4f} | Time: {elapsed:.1f}s")




In [ ]:
# ---------------------------
# Run
# ---------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DETR(num_classes=2).to(device)
matcher = HungarianMatcher()
dataset = CocoDetectionDataset('C:/Users/mza0288/Datset/train2017', 'C:/Users/mza0288/Datset/annotations/instances_train2017.json')
dataloader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=lambda x: tuple(zip(*x)), num_workers=4)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scheduler = None

for epoch in range(10):
    train_fn(dataloader, model, detr_loss, optimizer, device, scheduler, epoch, matcher)

torch.save(model.state_dict(), "custom_detr_instance_seg.pth")


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\mza0288/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth
100%|█████████████████████████████████████████████████████████████████████████████| 97.8M/97.8M [00:01<00:00, 55.8MB/s]
C:\Users\mza0288\AppData\Local\Temp\ipykernel_32052\2644269142.py:162: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


loading annotations into memory...
Done (t=0.09s)
creating index...
index created!


Epoch 1:   0%|                                                                                 | 0/208 [00:00<?, ?it/s]